# Cooling Technique Recommender Training

This notebook trains a multiclass model to predict `bestTechnique` using:
- `tempC`
- `rh`
- `itLoadKW`
- `electricityPrice`
- `waterPrice`
- `carbonFactor`

It is designed to run in Colab or VS Code notebooks.

## 1) Setup and Environment Check

In [ ]:
import os
import sys
import platform

print('Python:', sys.version)
print('Platform:', platform.platform())
print('Working dir:', os.getcwd())
print('Files in working dir (first 20):', os.listdir('.')[:20])

## 2) Install and Import Dependencies

In [ ]:
# If running in Colab, this installs required packages.
# In VS Code local environment, run once if missing.
%pip -q install pandas numpy scikit-learn seaborn matplotlib joblib xgboost

In [ ]:
import io
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import joblib

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)

## 3) Configure Runtime Parameters

In [ ]:
RANDOM_SEED = 42
TEST_SIZE = 0.20
TARGET_COL = 'bestTechnique'

FEATURE_COLS = [
    'tempC', 'rh', 'itLoadKW',
    'electricityPrice', 'waterPrice', 'carbonFactor'
]

LABEL_DATASET_CANDIDATES = [
    'dataset - Copy.csv',
    'dataset_copy.csv',
    'dataset.csv'
]

FULL_DATASET_CANDIDATES = [
    'dataset_full - Copy.csv',
    'dataset_full.csv'
]

ARTIFACT_MODEL_PATH = 'cooling_recommender_artifact.pkl'
ARTIFACT_METRICS_PATH = 'cooling_recommender_metrics.json'

print('Target:', TARGET_COL)
print('Features:', FEATURE_COLS)

## 4) Load Input Data

In [ ]:
def _find_file(candidates, available_names):
    lower_map = {name.lower(): name for name in available_names}
    for c in candidates:
        if c.lower() in lower_map:
            return lower_map[c.lower()]
    for name in available_names:
        for c in candidates:
            if c.lower() in name.lower():
                return name
    return None

# Try Colab upload first
try:
    from google.colab import files
    print('Colab detected. Please upload dataset CSV files...')
    uploaded = files.upload()
    uploaded_names = list(uploaded.keys())
    label_file = _find_file(LABEL_DATASET_CANDIDATES, uploaded_names)
    full_file = _find_file(FULL_DATASET_CANDIDATES, uploaded_names)

    if label_file is None:
        raise ValueError('Could not find label dataset in uploads.')

    df = pd.read_csv(io.BytesIO(uploaded[label_file]))
    df_full = pd.read_csv(io.BytesIO(uploaded[full_file])) if full_file else None
    print(f'Using uploaded label dataset: {label_file}')
    if full_file:
        print(f'Using uploaded full dataset: {full_file}')

except Exception:
    # Local notebook fallback
    print('Using local file fallback mode...')
    import os
    local_files = os.listdir('.')
    label_file = _find_file(LABEL_DATASET_CANDIDATES, local_files)
    full_file = _find_file(FULL_DATASET_CANDIDATES, local_files)

    if label_file is None:
        raise FileNotFoundError('Label dataset not found locally. Place dataset - Copy.csv in current folder.')

    df = pd.read_csv(label_file)
    df_full = pd.read_csv(full_file) if full_file else None
    print(f'Using local label dataset: {label_file}')
    if full_file:
        print(f'Using local full dataset: {full_file}')

print('Label dataset shape:', df.shape)
if df_full is not None:
    print('Full dataset shape:', df_full.shape)

display(df.head())
print('\nColumns:', list(df.columns))

## 5) Clean and Validate Data

In [ ]:
# Validate required columns
missing_required = [c for c in FEATURE_COLS + [TARGET_COL] if c not in df.columns]
assert len(missing_required) == 0, f'Missing required columns: {missing_required}'

print('Missing values:')
print(df[FEATURE_COLS + [TARGET_COL]].isna().sum())

print('\nDuplicate rows:', df.duplicated().sum())

work = df[FEATURE_COLS + [TARGET_COL]].copy()
work = work.drop_duplicates()

for c in FEATURE_COLS:
    work[c] = pd.to_numeric(work[c], errors='coerce')

work[TARGET_COL] = work[TARGET_COL].astype(str).str.strip()
work = work.dropna(subset=FEATURE_COLS + [TARGET_COL])

# Basic class validation
valid_classes = {'AirEconomizer', 'Evaporative', 'ChilledWater'}
unknown = set(work[TARGET_COL].unique()) - valid_classes
print('Observed classes:', sorted(work[TARGET_COL].unique()))
if unknown:
    print('Warning: unexpected labels found:', unknown)

print('\nClass distribution:')
print(work[TARGET_COL].value_counts())

plt.figure(figsize=(6,4))
sns.countplot(data=work, x=TARGET_COL, order=work[TARGET_COL].value_counts().index)
plt.title('Class Distribution')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

print('Cleaned shape:', work.shape)

## 6) Feature Engineering

**Leakage warning:** do not use `air_*`, `evap_*`, `chill_*` columns from full dataset for production recommender training, because those are outcome variables.

In [ ]:
X = work[FEATURE_COLS].copy()
y = work[TARGET_COL].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=y
)

print('Train:', X_train.shape, 'Test:', X_test.shape)

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, FEATURE_COLS)
], remainder='drop')

## 7) Train Baseline Models

In [ ]:
baseline_models = {
    'LogReg': Pipeline([
        ('prep', preprocessor),
        ('clf', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_SEED))
    ]),
    'RandomForest': Pipeline([
        ('prep', preprocessor),
        ('clf', RandomForestClassifier(
            n_estimators=500,
            min_samples_split=4,
            class_weight='balanced',
            random_state=RANDOM_SEED,
            n_jobs=-1
        ))
    ])
}

# Optional XGBoost
try:
    from xgboost import XGBClassifier
    baseline_models['XGBoost'] = Pipeline([
        ('prep', preprocessor),
        ('clf', XGBClassifier(
            n_estimators=500,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            objective='multi:softprob',
            eval_metric='mlogloss',
            random_state=RANDOM_SEED
        ))
    ])
except Exception as ex:
    print('XGBoost not available:', ex)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
cv_rows = []
for name, model in baseline_models.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='f1_macro', n_jobs=-1)
    cv_rows.append({'model': name, 'cv_macro_f1_mean': scores.mean(), 'cv_macro_f1_std': scores.std()})
    print(f"{name}: Macro-F1 = {scores.mean():.4f} ± {scores.std():.4f}")

cv_df = pd.DataFrame(cv_rows).sort_values('cv_macro_f1_mean', ascending=False).reset_index(drop=True)
display(cv_df)

## 8) Evaluate Model Performance

In [ ]:
# Tune RandomForest as stable production candidate
rf_pipe = baseline_models['RandomForest']

param_dist = {
    'clf__n_estimators': [300, 500, 700, 900],
    'clf__max_depth': [None, 10, 16, 24, 32],
    'clf__min_samples_split': [2, 4, 6, 10],
    'clf__min_samples_leaf': [1, 2, 4],
    'clf__max_features': ['sqrt', 'log2', None]
}

search = RandomizedSearchCV(
    estimator=rf_pipe,
    param_distributions=param_dist,
    n_iter=20,
    scoring='f1_macro',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED),
    n_jobs=-1,
    random_state=RANDOM_SEED,
    verbose=1
)
search.fit(X_train, y_train)

final_model = search.best_estimator_
print('Best CV Macro-F1:', search.best_score_)
print('Best params:', search.best_params_)

pred = final_model.predict(X_test)
acc = accuracy_score(y_test, pred)
macro_f1 = f1_score(y_test, pred, average='macro')
print(f'\nTest Accuracy: {acc:.4f}')
print(f'Test Macro-F1: {macro_f1:.4f}')
print('\nClassification report:')
print(classification_report(y_test, pred, digits=4))

labels = sorted(y.unique())
cm = confusion_matrix(y_test, pred, labels=labels)
plt.figure(figsize=(7,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

# Feature importance (RF)
rf_model = final_model.named_steps['clf']
if hasattr(rf_model, 'feature_importances_'):
    importances = pd.Series(rf_model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
    display(importances)
    plt.figure(figsize=(6,4))
    sns.barplot(x=importances.values, y=importances.index)
    plt.title('Feature Importances')
    plt.tight_layout()
    plt.show()

## 9) Persist Outputs and Artifacts

In [ ]:
artifact = {
    'model': final_model,
    'feature_cols': FEATURE_COLS,
    'target_col': TARGET_COL,
    'random_seed': RANDOM_SEED
}
joblib.dump(artifact, ARTIFACT_MODEL_PATH)

metrics = {
    'test_accuracy': float(acc),
    'test_macro_f1': float(macro_f1),
    'n_rows': int(len(work)),
    'class_distribution': work[TARGET_COL].value_counts().to_dict(),
    'best_params': search.best_params_
}
with open(ARTIFACT_METRICS_PATH, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

print('Saved model:', ARTIFACT_MODEL_PATH)
print('Saved metrics:', ARTIFACT_METRICS_PATH)

try:
    from google.colab import files
    files.download(ARTIFACT_MODEL_PATH)
    files.download(ARTIFACT_METRICS_PATH)
except Exception:
    print('Colab download skipped (local mode).')

In [ ]:
def recommend_best_technique(tempC, rh, itLoadKW, electricityPrice, waterPrice, carbonFactor, model_artifact=artifact):
    model = model_artifact['model']
    cols = model_artifact['feature_cols']
    row = pd.DataFrame([{
        'tempC': tempC,
        'rh': rh,
        'itLoadKW': itLoadKW,
        'electricityPrice': electricityPrice,
        'waterPrice': waterPrice,
        'carbonFactor': carbonFactor
    }])[cols]
    pred = model.predict(row)[0]
    if hasattr(model, 'predict_proba'):
        probs = model.predict_proba(row)[0]
        labels = model.classes_
        prob_map = dict(zip(labels, probs))
        return pred, prob_map
    return pred, None

pred_label, pred_probs = recommend_best_technique(32, 55, 1200, 0.14, 1.2, 0.45)
print('Example recommendation:', pred_label)
print('Class probabilities:', pred_probs)

In [ ]:
# Reproducibility quick rerun summary
print('--- Reproducibility Summary ---')
print('Rows used:', len(work))
print('Test Accuracy:', round(acc, 4))
print('Test Macro-F1:', round(macro_f1, 4))
print('Model saved to:', ARTIFACT_MODEL_PATH)
print('Metrics saved to:', ARTIFACT_METRICS_PATH)

## 10) Backend-Only Explanation Output (Non-Technical)

This section generates backend-ready response text with:
- one best technique,
- clear comparison reasons explaining why it was selected,
- future impact in paragraph form (not raw technical table).

Use this after model training. No frontend integration is required.